# 🏦 Mini-Project: Enterprise ATM Banking System

### *Evolution from Procedural Python ➔ Production-Ready Object-Oriented Architecture*

---

## 🚀 What's New in Version 4 (V4)?
Aapne **Day 14 (File Handling)** aur **Day 15 (OOPs)** me jo naye concepts seekhe hain, un sabhi ko is Mini-Project me integrate kiya gaya hai:

| Core Concept | Day Reference | Real-World Application in ATM V4 |
|---|---|---|
| **Encapsulation** | Day 15 (Module 04) | Sensitive data (`__pin`, `_balance`) ko private kiya gaya hai with `@property` |
| **Abstraction** | Day 15 (Module 05) | `BaseAccount(ABC)` abstract class banayi gayi hai jo contract enforce karti hai |
| **Inheritance & Polymorphism** | Day 15 (Modules 02 & 03) | `SavingsAccount` aur `CurrentAccount` alag-alag withdrawal rules ke sath inherit karte hain |
| **Magic / Dunder Methods** | Day 15 (Module 06) | `__str__` (formatted slip), `__repr__` (debugging), aur `__len__` (transaction count) |
| **Custom Exceptions** | Day 15 (Module 08) | `InsufficientFundsError`, `InvalidPinError`, `InvalidAmountError` for robust error handling |
| **File Handling & Persistence** | Day 14 (File I/O) | Account balance `accounts.json` me save hota hai aur transaction log `atm_transactions.log` me append hota hai |


---
## 📜 Section 1: Previous ATM Machine V3 (Procedural Approach)


In [ ]:
# ==========================================
# 📜 ATM Machine V3 (Procedural Reference)
# ==========================================
# Concepts: Global variables, Functions, if-elif-else, while loop

v3_balance = 5000
v3_transactions = 0

def v3_show_menu():
    print("\n" + "=" * 35)
    print("   🏦 ATM SYSTEM V3 (Procedural)")
    print("=" * 35)
    print("1. Check Balance\n2. Deposit\n3. Withdraw\n4. Exit")
    print("=" * 35)

print("ATM V3 Reference loaded. (Run V4 below for full OOP + Persistence edition!)")


---
## 🏛️ Section 2: ATM Machine V4 (Enterprise OOP + File Persistence)

Neeche diye gaye code me saari classes, custom exceptions, abstract interfaces, aur file logging implement ki gayi hain:

In [7]:
from abc import ABC, abstractmethod
from datetime import datetime
import os
import json


class ATMError(Exception):
    """Base exception for all ATM errors"""
    pass

class InsufficientFundsError(ATMError):
    def __init__(self, current_balance, requested_amount):
        super().__init__(f"Insufficient Funds! Available: Rs. {current_balance:,.2f}, Requested: Rs. {requested_amount:,.2f}")
        self.current_balance = current_balance
        self.requested_amount = requested_amount

class InvalidPinError(ATMError):
    pass

class InvalidAmountError(ATMError):
    pass



class BaseAccount(ABC):
    """Abstract base class defining the bank account contract"""
    def __init__(self, account_no, holder_name, initial_balance=0.0):
        self._account_no = str(account_no)
        self._holder_name = str(holder_name)
        self._balance = float(initial_balance)
        self._history = []

    @property
    def balance(self):
        """Read-only property for encapsulated balance"""
        return self._balance

    @property
    def account_no(self):
        return self._account_no

    @property
    def holder_name(self):
        return self._holder_name

    @abstractmethod
    def deposit(self, amount):
        pass

    @abstractmethod
    def withdraw(self, amount):
        pass



class SavingsAccount(BaseAccount):
    def __init__(self, account_no, holder_name, initial_balance=5000.0, pin="1234", min_balance=500.0):
        super().__init__(account_no, holder_name, initial_balance)
        self.__pin = str(pin)
        self.min_balance = float(min_balance)

    def authenticate(self, entered_pin):
        return self.__pin == str(entered_pin)

    def update_pin(self, old_pin, new_pin):
        if not self.authenticate(old_pin):
            raise InvalidPinError("Authentication failed: Current PIN is incorrect!")
        if len(str(new_pin)) != 4 or not str(new_pin).isdigit():
            raise InvalidPinError("New PIN must be exactly 4 digits!")
        self.__pin = str(new_pin)
        return True

    def deposit(self, amount):
        amount = float(amount)
        if amount <= 0:
            raise InvalidAmountError("Deposit amount must be positive!")
        self._balance += amount
        self._history.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "action": "DEPOSIT",
            "amount": amount,
            "balance": self._balance
        })
        return self._balance

    def withdraw(self, amount):
        amount = float(amount)
        if amount <= 0:
            raise InvalidAmountError("Withdrawal amount must be positive!")
        if (self._balance - amount) < self.min_balance:
            raise InsufficientFundsError(self._balance, amount)
        self._balance -= amount
        self._history.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "action": "WITHDRAW",
            "amount": amount,
            "balance": self._balance
        })
        return self._balance

    def transfer(self , target_acc , amount):

        self.withdraw(amount)

        target_acc.deposit(amount)

        print(f"Transferred Rs. {amount} to {target_acc.holder_name} successfully!")


    
    def __str__(self):
        return f"SavingsAccount [Acc: {self._account_no} | Holder: {self._holder_name} | Balance: Rs. {self._balance:,.2f}]"

    def __repr__(self):
        return f"SavingsAccount(acc='{self._account_no}', holder='{self._holder_name}', balance={self._balance})"

    def __len__(self):
        return len(self._history)



class ATMDataStore:
    """Handles permanent file storage and transaction log streams"""
    def __init__(self, data_file="accounts_db.json", log_file="atm_transactions.log"):
        self.data_file = data_file
        self.log_file = log_file

    def save_account_state(self, account: SavingsAccount):
        """Writes account balance and metadata to JSON file"""
        state = {
            "account_no": account.account_no,
            "holder_name": account.holder_name,
            "balance": account.balance,
            "total_transactions": len(account),
            "last_synced": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        with open(self.data_file, "w", encoding="utf-8") as f:
            json.dump(state, f, indent=4)

    def load_or_create_account(self, default_account: SavingsAccount) -> SavingsAccount:
        """Restores previous balance from disk, ensuring data is never lost!"""
        if os.path.exists(self.data_file):
            try:
                with open(self.data_file, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    acc = SavingsAccount(
                        account_no=data.get("account_no", default_account.account_no),
                        holder_name=data.get("holder_name", default_account.holder_name),
                        initial_balance=data.get("balance", default_account.balance)
                    )
                    return acc
            except Exception:
                return default_account
        return default_account

    def append_audit_log(self, account_no, action, amount, resulting_balance):
        """Appends chronological transaction record using Day 14 'a' mode"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = (
            f"[{timestamp}] ACC: {account_no} | "
            f"{action:<8} | Amount: Rs. {amount:>10.2f} | "
            f"Balance: Rs. {resulting_balance:>12.2f}\n"
        )
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(log_entry)

    def get_recent_logs(self, limit=5):
        """Reads the last N lines from the transaction audit log"""
        if not os.path.exists(self.log_file):
            return ["No transactions recorded yet."]
        with open(self.log_file, "r", encoding="utf-8") as f:
            lines = f.readlines()
            return [line.strip() for line in lines[-limit:]]



class ATMController:
    def __init__(self, account: SavingsAccount, store: ATMDataStore):
        self.store = store
        self.account = self.store.load_or_create_account(account)
        self.border = "=" * 50

    def display_menu(self):
        print("\n" + self.border)
        print("       🏦 NEXT-GEN PYTHON ATM SYSTEM (V4)")
        print("       Powered by OOPs & File Persistence")
        print(self.border)
        print("1. 💰 Check Balance")
        print("2. ➕ Deposit Money")
        print("3. ➖ Withdraw Money")
        print("4. 📜 Mini-Statement (Audit Log)")
        print("5. 🔐 Change Security PIN")
        print("6. 🚪 Exit & Save")
        print(self.border)

    def handle_deposit(self, amount):
        try:
            new_balance = self.account.deposit(amount)
            self.store.append_audit_log(self.account.account_no, "DEPOSIT", amount, new_balance)
            self.store.save_account_state(self.account)
            print(f"\n✅ Deposit Successful: +Rs. {amount:,.2f}")
            print(f"💰 Current Balance: Rs. {new_balance:,.2f}")
        except InvalidAmountError as e:
            print(f"\n❌ Error: {e}")

    def handle_withdraw(self, amount):
        try:
            new_balance = self.account.withdraw(amount)
            self.store.append_audit_log(self.account.account_no, "WITHDRAW", amount, new_balance)
            self.store.save_account_state(self.account)
            print(f"\n✅ Withdrawal Successful: -Rs. {amount:,.2f}")
            print(f"💰 Remaining Balance: Rs. {new_balance:,.2f}")
        except InsufficientFundsError as e:
            print(f"\n❌ {e}")
        except InvalidAmountError as e:
            print(f"\n❌ Error: {e}")

    def print_mini_statement(self):
        print("\n" + self.border)
        print(f"📜 RECENT TRANSACTIONS (Account #{self.account.account_no})")
        print(self.border)
        logs = self.store.get_recent_logs(limit=5)
        for log in logs:
            print(log)
        print(self.border)

print("✅ ATM System V4 Architecture compiled successfully!")


✅ ATM System V4 Architecture compiled successfully!


In [8]:
# 1. Shivaji ka account (Sender)
shivaji_acc = SavingsAccount(account_no="1001", holder_name="Shivaji", initial_balance=6300.0)

# 2. Rahul ka account (Receiver / Target Account)
rahul_acc = SavingsAccount(account_no="1002", holder_name="Rahul", initial_balance=1000.0)

print("--- Transfer se Pehle ---")
print(shivaji_acc)
print(rahul_acc)

# 3. Shivaji ne Rahul ko Rs. 1,500 transfer kiye
print("\n--- Transfer Process ---")
shivaji_acc.transfer(target_acc=rahul_acc, amount=1500)

print("\n--- Transfer ke Baad ---")
print(shivaji_acc)  # Iska balance Rs. 4,800 ho jayega
print(rahul_acc)    # Iska balance Rs. 2,500 ho jayega


--- Transfer se Pehle ---
SavingsAccount [Acc: 1001 | Holder: Shivaji | Balance: Rs. 6,300.00]
SavingsAccount [Acc: 1002 | Holder: Rahul | Balance: Rs. 1,000.00]

--- Transfer Process ---
Transferred Rs. 1500 to Rahul successfully!

--- Transfer ke Baad ---
SavingsAccount [Acc: 1001 | Holder: Shivaji | Balance: Rs. 4,800.00]
SavingsAccount [Acc: 1002 | Holder: Rahul | Balance: Rs. 2,500.00]


---
## 🧪 Section 3: Live Demonstration & Automatic Testing

Run this cell to test the full lifecycle: creating accounts, deposits, withdrawals, custom exceptions, file persistence, and audit logging!

In [2]:
# Initialize DataStore and Account
store = ATMDataStore(data_file="accounts_db.json", log_file="atm_transactions.log")
default_acc = SavingsAccount(account_no="1001", holder_name="Shivaji", initial_balance=5000.0, pin="4321")

atm = ATMController(default_acc, store)

print("1. Account String Representation (__str__):")
print(atm.account)

print("\n2. Performing Deposit:")
atm.handle_deposit(2500)

print("\n3. Performing Successful Withdrawal:")
atm.handle_withdraw(1200)

print("\n4. Testing Custom Exception (InsufficientFundsError):")
atm.handle_withdraw(99999)  # Should trigger custom exception gracefully

print("\n5. Displaying Mini Statement from File Audit Log:")
atm.print_mini_statement()

print(f"\n6. Total Session Transactions (__len__): {len(atm.account)}")


1. Account String Representation (__str__):
SavingsAccount [Acc: 1001 | Holder: Shivaji | Balance: Rs. 5,000.00]

2. Performing Deposit:

✅ Deposit Successful: +Rs. 2,500.00
💰 Current Balance: Rs. 7,500.00

3. Performing Successful Withdrawal:

✅ Withdrawal Successful: -Rs. 1,200.00
💰 Remaining Balance: Rs. 6,300.00

4. Testing Custom Exception (InsufficientFundsError):

❌ Insufficient Funds! Available: Rs. 6,300.00, Requested: Rs. 99,999.00

5. Displaying Mini Statement from File Audit Log:

📜 RECENT TRANSACTIONS (Account #1001)
[2026-10-15 18:23:33] ACC: 1001 | DEPOSIT  | Amount: Rs.    2500.00 | Balance: Rs.      7500.00
[2026-10-15 18:23:33] ACC: 1001 | WITHDRAW | Amount: Rs.    1200.00 | Balance: Rs.      6300.00

6. Total Session Transactions (__len__): 2


---
### 🏆 Key Improvements Summary (V3 vs V4):

| Feature | V3 (Procedural) | V4 (Enterprise OOP) |
|---|---|---|
| **Data State** | Lost on program exit | Saved permanently in `accounts_db.json` |
| **History** | Counter only (`transactions += 1`) | Timestamped Audit Trail in `atm_transactions.log` |
| **Security** | Global variable `balance` editable by anyone | Encapsulated `__pin` and protected `_balance` with `@property` |
| **Error Handling** | Basic `print("Error")` | Dedicated Custom Exceptions (`InsufficientFundsError`, etc.) |
| **Extensibility** | Hard to add new account types | Subclass `BaseAccount` to add `CurrentAccount`, `CreditAccount` easily |
